In [61]:
import requests
import json
import pandas

In [8]:
BASE_URL = "https://futamina.com/api/v1/feeds/virtual"
KEY = "14e402b7447b927785aea6b9a8ec8ccdc49c6eb4"

In [8]:
def get_leagues(base_url):
    get_league_url=base_url+"/leagues"
    get_leagues_result = requests.get(get_league_url, headers={"X-Api-Key": KEY})
    return get_leagues_result.json()

In [9]:
all_leagues = get_leagues(base_url=BASE_URL)
print(all_leagues)

{'leagues': [{'slug': 'virtual-kiron-mundial', 'name': 'Mundial'}, {'slug': 'virtual-kiron-brasil', 'name': 'Brasil'}, {'slug': 'virtual-kiron-inglaterra', 'name': 'Inglaterra'}, {'slug': 'virtual-kiron-italia', 'name': 'Itália'}, {'slug': 'virtual-kiron-latam', 'name': 'América Latina (Kiron)'}, {'slug': 'virtual-kiron-espanha', 'name': 'Espanha'}, {'slug': 'virtual-inspired-champions', 'name': 'Champions'}, {'slug': 'virtual-inspired-international', 'name': 'Internacional'}, {'slug': 'virtual-inspired-soccer3', 'name': 'América Latina (Inspired)'}]}


In [10]:
def get_results_by_league(slug):
    get_results_by_league_link = BASE_URL+f"/leagues/{slug}/results"
    get_results_by_league_results = requests.get(get_results_by_league_link, headers={"X-Api-Key": KEY})
    return_data = get_results_by_league_results.json()
    # print (json.dumps(return_data, indent=2, ensure_ascii=False))
    return return_data

In [52]:
all_games = get_results_by_league('virtual-kiron-mundial')

In [60]:
len(all_games['results'])

685

In [40]:
for league in all_leagues["leagues"]:
    # print(league["slug"])
    get_results_by_league(league["slug"])


{
  "league": {
    "slug": "virtual-kiron-mundial",
    "name": "Mundial"
  },
  "results": [
    {
      "id": "540f43ae-b9d4-45e4-a6bb-618b98cdb590",
      "startsAt": "2026-11-10T00:42:00.000Z",
      "home": "FRA",
      "away": "POL",
      "score": "1-0",
      "homeGoals": 1,
      "awayGoals": 0
    },
    {
      "id": "702e4d3b-0ee3-4463-b764-b5476058d046",
      "startsAt": "2026-11-10T00:40:00.000Z",
      "home": "WAL",
      "away": "CRO",
      "score": "2-1",
      "homeGoals": 2,
      "awayGoals": 1
    },
    {
      "id": "7a497a0a-bccb-4933-8925-cc0dd5fea895",
      "startsAt": "2026-11-10T00:38:00.000Z",
      "home": "SPA",
      "away": "DEN",
      "score": "2-1",
      "homeGoals": 2,
      "awayGoals": 1
    },
    {
      "id": "b48591f6-e2fa-4ac3-8e31-2e5eda9c040f",
      "startsAt": "2026-11-10T00:36:00.000Z",
      "home": "ENG",
      "away": "SWI",
      "score": "2-0",
      "homeGoals": 2,
      "awayGoals": 0
    },
    {
      "id": "fb2e4df5-c9dd-

In [15]:
def find_over_goals_patterns(slug=None, min_rate=0.9, skip_limbo=1, next_games=3, min_samples=10):
    """Procura placares que antecedem mais de 2 gols nos jogos seguintes.

    Depois do placar gatilho, pula `skip_limbo` jogo e olha os `next_games`
    seguintes. O placar entra na lista quando, em pelo menos `min_rate` das
    ocorrencias com amostra minima `min_samples`, algum desses jogos fecha
    com soma de gols maior que 2.

    Sem `slug`, percorre todas as ligas via get_results_by_league().
    """
    if slug is None:
        slugs = [league["slug"] for league in get_leagues(BASE_URL)["leagues"]]
    else:
        slugs = [slug]

    patterns = []
    for league_slug in slugs:
        payload = get_results_by_league(league_slug)
        league = payload.get("league", {})
        played = [
            game for game in payload.get("results", [])
            if game.get("homeGoals") is not None and game.get("awayGoals") is not None
        ]
        played.sort(key=lambda game: game["startsAt"])

        def placar(game):
            return f"{game['homeGoals']}x{game['awayGoals']}"

        def total_goals(game):
            return game["homeGoals"] + game["awayGoals"]

        stats = {}
        for index, game in enumerate(played):
            window_start = index + skip_limbo + 1
            window_end = window_start + next_games
            if window_end > len(played):
                break
            window = played[window_start:window_end]
            hits_in_window = sum(total_goals(item) > 2 for item in window)
            bucket = stats.setdefault(placar(game), {"amostras": 0, "acertos": 0})
            bucket["amostras"] += 1
            bucket["acertos"] += int(hits_in_window >= 1)

        latest_score = placar(played[-1]) if played else None
        previous_score = placar(played[-2]) if len(played) >= 2 else None

        for score, bucket in stats.items():
            if bucket["amostras"] < min_samples:
                continue
            rate = bucket["acertos"] / bucket["amostras"]
            if rate < min_rate:
                continue

            if score == previous_score and score == latest_score:
                situacao = "limbo_passou_e_novo_gatilho"
                proximo_passo = "o ultimo jogo foi limbo e tambem um novo gatilho; os proximos 3 sao a entrada"
            elif score == previous_score:
                situacao = "limbo_passou"
                proximo_passo = "o limbo ja passou; os proximos 3 jogos sao a entrada"
            elif score == latest_score:
                situacao = "gatilho_agora"
                proximo_passo = "o proximo jogo e limbo; a entrada sao os 3 jogos depois dele"
            else:
                situacao = "historico"
                proximo_passo = "padrao valido no historico; o ultimo jogo nao disparou esse placar"

            patterns.append({
                "liga": league.get("slug", league_slug),
                "liga_nome": league.get("name"),
                "placar": score,
                "amostras": bucket["amostras"],
                "acertos": bucket["acertos"],
                "taxa": round(rate, 4),
                "situacao": situacao,
                "proximo_passo": proximo_passo,
                "regra": (
                    f"depois de {score}, pular {skip_limbo} jogo (limbo); "
                    f"em pelo menos 1 dos {next_games} seguintes a soma de gols e maior que 2"
                ),
            })

    patterns.sort(key=lambda item: (-item["taxa"], -item["amostras"], item["liga_nome"] or "", item["placar"]))
    return patterns


In [16]:
# for league in all_leagues["leagues"]:
#     # print(league["slug"])
#     get_results_by_league(league["slug"])

# all_leagues["leagues"][0]["slug"]
find_over_goals_patterns(slug=all_leagues["leagues"][0]["slug"])

[{'liga': 'virtual-kiron-mundial',
  'liga_nome': 'Mundial',
  'placar': '2x3',
  'amostras': 15,
  'acertos': 15,
  'taxa': 1.0,
  'situacao': 'historico',
  'proximo_passo': 'padrao valido no historico; o ultimo jogo nao disparou esse placar',
  'regra': 'depois de 2x3, pular 1 jogo (limbo); em pelo menos 1 dos 3 seguintes a soma de gols e maior que 2'},
 {'liga': 'virtual-kiron-mundial',
  'liga_nome': 'Mundial',
  'placar': '4x2',
  'amostras': 13,
  'acertos': 13,
  'taxa': 1.0,
  'situacao': 'historico',
  'proximo_passo': 'padrao valido no historico; o ultimo jogo nao disparou esse placar',
  'regra': 'depois de 4x2, pular 1 jogo (limbo); em pelo menos 1 dos 3 seguintes a soma de gols e maior que 2'},
 {'liga': 'virtual-kiron-mundial',
  'liga_nome': 'Mundial',
  'placar': '0x1',
  'amostras': 57,
  'acertos': 52,
  'taxa': 0.9123,
  'situacao': 'historico',
  'proximo_passo': 'padrao valido no historico; o ultimo jogo nao disparou esse placar',
  'regra': 'depois de 0x1, pular 

In [ ]:
padroes = find_over_goals_patterns()
print(json.dumps(padroes, indent=2, ensure_ascii=False))
